# Hybrid RAG: Vector Search + Knowledge Graph

## What is RAG?

**RAG (Retrieval-Augmented Generation)** is a technique used in AI systems where, instead of relying solely on the knowledge a language model learned during training, we *retrieve* relevant information from an external knowledge base at query time — and then feed that information to the model alongside the question.

This makes AI answers:
- **More accurate** — grounded in real, verifiable data (no hallucinations)
- **More up-to-date** — can use data newer than the model's training cutoff
- **More explainable** — we know exactly which documents the answer came from

---

## Why Hybrid? Vector Search + Graph Search

Most RAG systems use only vector search. This project goes further by combining **two complementary retrieval strategies**:

| Strategy | Technology | What It Finds |
|----------|-----------|---------------|
| **Semantic Search** | Pinecone (vector DB) | Text chunks *semantically similar* to the query |
| **Graph Traversal** | Neo4j (graph DB) | *Structurally connected* entities and sections |

**Why combine them?**  
Vector search finds *what* is relevant. Graph search finds *how things are connected*. Together, the LLM receives both the relevant text chunk AND a map of related entities — far richer context for reasoning.

---

## How It Works

**Ingestion (one-time setup, Steps 1–5):**
1. JSON documents are loaded and key entities are stored as Neo4j nodes
2. Relationships between entities are established in the graph
3. The same text is split into overlapping chunks, embedded via OpenAI, and stored in Pinecone

**Retrieval (at query time, Steps 6–7):**
1. The user's question is embedded using the same OpenAI model
2. Pinecone finds the top-k chunks whose embeddings are closest to the question
3. For each matching chunk, we use the entity name to query Neo4j for all related nodes
4. The result contains both the matching text chunk AND its graph neighborhood

---

## Dataset

We use three Wikipedia-derived JSON files about interconnected historical topics:

| File | Node Type | About |
|------|-----------|-------|
| `Napoleon.json` | Person | French Emperor and military strategist (1769–1821) |
| `Talleyrand.json` | Person | French diplomat, closely allied with Napoleon |
| `Battle_of_Waterloo.json` | Event | The 1815 battle that ended Napoleon's reign |

These are intentionally interconnected — making graph traversal meaningful and demonstrating the power of hybrid retrieval.

---

## Step 1 — Connect to Databases

In [ ]:
# Import the function that creates Neo4j nodes (defined in ingest/neo4j.py)
from ingest.neo4j import create_nodes

# Import the Neo4j config module — it reads credentials from .env and
# returns a LangChain Neo4jGraph client (a convenient wrapper around the
# native Neo4j driver that also supports Cypher queries via .query())
from config import neo4j

import json  # Standard library — used to parse JSON data files

# Establish the connection to Neo4j.
# This reads NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD from your .env file.
# If the connection fails, check that your Neo4j instance is running
# and that your .env credentials are correct.
graph = neo4j.load_neo4j_graph()

## Step 2 — Define the Data Sources

Our knowledge base consists of three JSON files in the `data/` directory. Each file represents one entity and is structured as a flat dictionary:

```json
{
  "General Information": "Napoleon Bonaparte was a French military...",
  "Career": "He rose to prominence during the French Revolution...",
  "Death": "Napoleon died on 5 May 1821 at Saint Helena..."
}
```

Each **key** is a section name — this becomes a `Section` node in Neo4j and a metadata label in Pinecone.  
Each **value** is the raw text for that section — this is what gets embedded and stored in Pinecone.

We process three entities:
- `Napoleon` → labeled as a `Person` node in Neo4j
- `Talleyrand` → labeled as a `Person` node in Neo4j
- `Battle_of_Waterloo` → labeled as an `Event` node in Neo4j

In [ ]:
# The names of the three entities we want to ingest.
# These correspond to filenames: data/Napoleon.json, data/Talleyrand.json, etc.
# They are also used as node names in Neo4j (with "_info" appended, e.g. "Napoleon_info").
file_names = ["Talleyrand", "Napoleon", "Battle_of_Waterloo"]

## Step 3 — Ingest into Neo4j: Create Nodes

### What is Neo4j?
Neo4j is a **graph database** — instead of storing data in tables (like SQL), it stores data as a network of **nodes** (entities) and **relationships** (directed edges between entities). This makes it ideal for questions like *"What is everything connected to Napoleon?"*

### What `create_nodes()` Does
For each entity, it runs two types of Cypher `MERGE` statements:

**1. Main node** — represents the whole entity:
```
(Napoleon_info :Person)
(Talleyrand_info :Person)
(Battle_of_Waterloo_info :Event)
```

**2. Section nodes** — one per section key from the JSON file:
```
(:Section {type: "General Information", parent_name: "Napoleon_info"})
(:Section {type: "Career",              parent_name: "Napoleon_info"})
(:Section {type: "Death",               parent_name: "Napoleon_info"})
```

### Why `MERGE` Instead of `CREATE`?
`MERGE` means *"create this node only if it doesn't already exist"*. This makes the ingestion **idempotent** — you can safely run this notebook multiple times without creating duplicate nodes.

### Node Labels: `Person` vs `Event`
In Neo4j, a **label** acts like a type tag. Using separate labels (`Person`, `Event`) lets us write targeted Cypher queries later — for example, "find all Person nodes" or "connect all Event nodes to their sections".

In [ ]:
for name in file_names:
    # Build the path to the JSON file for this entity
    file = f"Data/{name}.json"

    # Load the JSON — structure is {section_name: section_text}
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Assign the appropriate Neo4j node label.
    # In the graph, labels act as types — they let us distinguish between
    # people and events in Cypher queries (e.g. MATCH (p:Person) vs MATCH (e:Event)).
    if name == "Battle_of_Waterloo":
        # Battles are Events — not people
        create_nodes(graph=graph, data=data, node_label="Event", node_name=name)
    else:
        # Napoleon and Talleyrand are Persons
        create_nodes(graph=graph, data=data, node_label="Person", node_name=name)

    print(f"Nodes created for: {name}")

## Step 4 — Create Relationships in the Graph

Now that our nodes exist, we connect them with **relationships** (the edges of the graph). In Neo4j, every relationship has:
- A **direction** — `(A)-[r]->(B)`
- A **type** — the label on the arrow, e.g. `RELATED_TO` or `HAS_SECTION`

### Relationships We Create

| Relationship | Connects | Example |
|---|---|---|
| `RELATED_TO` | Person ↔ Person | Napoleon ↔ Talleyrand |
| `RELATED_TO` | Person ↔ Event | Napoleon ↔ Battle of Waterloo |
| `HAS_SECTION` | Person → Section | Napoleon → "Death" section |
| `HAS_SECTION` | Event → Section | Battle of Waterloo → "Commanders" section |

### Why Bidirectional `RELATED_TO`?
We create both `(A)-[:RELATED_TO]->(B)` and `(B)-[:RELATED_TO]->(A)`. This is important because at retrieval time, we don't know *which direction* to traverse the graph from — so having both directions ensures we never miss a connection.

### Cypher Quick Reference
- `MATCH` — find existing nodes (like `SELECT` in SQL)
- `WHERE` — filter the results
- `MERGE` — create the relationship only if it doesn't already exist
- `elementId(p1) < elementId(p2)` — ensures we only create one pair per combination (avoids processing the same two nodes twice in the Person-Person query)

After running this cell, the graph looks like:
```
Napoleon_info ──[RELATED_TO]──► Talleyrand_info
Napoleon_info ──[RELATED_TO]──► Battle_of_Waterloo_info
Napoleon_info ──[HAS_SECTION]──► {type: "Death"}
Napoleon_info ──[HAS_SECTION]──► {type: "Career"}
... (and reverse directions)
```

In [ ]:
# ── Relationship 1: Person ↔ Person ──────────────────────────────────────────
# Connects every Person to every other Person (fully connected subgraph).
# elementId(p1) < elementId(p2) ensures we process each pair only once,
# but we still MERGE both directions so traversal works either way.
rel_person_person = """
MATCH (p1:Person), (p2:Person)
WHERE elementId(p1) < elementId(p2)
MERGE (p1)-[:RELATED_TO]->(p2)
MERGE (p2)-[:RELATED_TO]->(p1);
"""

# ── Relationship 2: Person ↔ Event ───────────────────────────────────────────
# Connects every Person to every Event. Since we only have one event
# (Battle of Waterloo), this links both Napoleon and Talleyrand to it.
rel_person_event = """
MATCH (p:Person), (e:Event)
MERGE (p)-[:RELATED_TO]->(e)
MERGE (e)-[:RELATED_TO]->(p);
"""

# ── Relationship 3: Person → Section ─────────────────────────────────────────
# Connects each Person node to its own Section nodes.
# The match condition `p.name = s.parent_name` is the join key —
# a Section's parent_name was set equal to the Person's name during node creation.
rel_person_section = """
MATCH (p:Person), (s:Section)
WHERE p.name = s.parent_name
MERGE (p)-[:HAS_SECTION]->(s);
"""

# ── Relationship 4: Event → Section ──────────────────────────────────────────
# Same as above, but for Event nodes (e.g. Battle_of_Waterloo_info → its sections).
rel_event_section = """
MATCH (e:Event), (s:Section)
WHERE e.name = s.parent_name
MERGE (e)-[:HAS_SECTION]->(s);
"""

# Run all four relationship queries against Neo4j
queries = [rel_person_person, rel_person_event, rel_person_section, rel_event_section]

for query in queries:
    graph.query(query)

print("All relationships created successfully.")

## Step 5 — Ingest into Pinecone: Vector Embeddings

### What are Vector Embeddings?
A **vector embedding** is a list of numbers (e.g. 1536 floats) that represents the *semantic meaning* of a piece of text. The key property: two texts that mean the same thing will have similar vectors — **even if they use completely different words**.

For example, these two sentences would produce very similar vectors:
- *"Napoleon died from stomach cancer"*
- *"Bonaparte's cause of death was a gastric illness"*

So a query about Napoleon's death would retrieve both — even though they share no words.

### How Pinecone Works
Pinecone stores vectors in a highly optimized **Approximate Nearest Neighbor (ANN) index** — designed to find the closest vectors to a query in milliseconds, even across millions of entries. The "closeness" is measured by **cosine similarity** (a score from 0 to 1, where 1 = identical meaning).

### What `process_and_upsert_files()` Does
For each JSON file, it:
1. **Splits** each section's text into overlapping chunks (~400 chars, 100-char overlap)  
   → Overlap ensures that context at chunk boundaries isn't cut off
2. **Embeds** each chunk using OpenAI's `text-embedding-3-small` model  
   → Produces a 1536-dimensional float vector per chunk
3. **Upserts** each vector into Pinecone with metadata:
   - `name` — entity name (e.g. `Napoleon_info`) — used to look up in Neo4j later
   - `section` — which section this chunk came from (e.g. `"Death"`)
   - `chunk_index` — position within the section
   - `text` — the original chunk text (returned in search results)

> **Why store the text in metadata?** Pinecone only stores vectors for search — to read the actual text back, we embed it in the metadata. This is a standard Pinecone pattern.

In [ ]:
# Import the ingestion function (defined in ingest/pinecone_ingest.py).
# This function handles chunking, embedding, and upserting all in one call.
# It uses the OpenAI client (from config/llm.py) and the Pinecone index
# (from config/pinecone_cfg.py + PINECONE_HOST env var) internally.
from ingest.pinecone_ingest import process_and_upsert_files

# Process and upsert all three files.
# Expected output: ~755 total chunks inserted (121 + 316 + 318).
# This may take a minute or two — each chunk requires an OpenAI API call to embed.
process_and_upsert_files(file_names)

## Step 6 — Define the Graph Traversal Query

This **Cypher query** defines how we fetch context from Neo4j once Pinecone has identified a relevant entity.

### Reading the Query
```cypher
MATCH (n)-[r]-(m)
WHERE n.name = $name
RETURN n AS matchedNode, r AS relationship, m AS relatedNode
```

Line by line:
- `MATCH (n)-[r]-(m)` — find any node `n` connected to another node `m` via relationship `r`. The dash-dash pattern `(n)-[r]-(m)` (no arrow) means **direction doesn't matter** — it traverses in either direction.
- `WHERE n.name = $name` — filter to the specific entity that Pinecone told us about (e.g. `"Napoleon_info"`). The `$name` is a **parameter** — we fill it in at query time.
- `RETURN` — give us back the matched node, each relationship object, and each neighboring node.

### What This Returns for `Napoleon_info`

| matchedNode | relationship | relatedNode |
|---|---|---|
| Napoleon_info | RELATED_TO | Talleyrand_info |
| Napoleon_info | RELATED_TO | Battle_of_Waterloo_info |
| Napoleon_info | HAS_SECTION | {type: "General Information"} |
| Napoleon_info | HAS_SECTION | {type: "Career"} |
| Napoleon_info | HAS_SECTION | {type: "Death"} |

This is the **graph neighborhood** — the full structural context that enriches our vector search result.

## Step 7 — Run the Hybrid Search

In [ ]:
# This Cypher query is parameterized with $name — at retrieval time,
# we substitute $name with the entity name returned by Pinecone
# (e.g. "Napoleon_info"). It fetches the 1-hop neighborhood:
# every node directly connected to the matched entity, plus the
# relationship type connecting them.
#
# Direction-agnostic pattern (n)-[r]-(m) ensures we traverse in
# both directions, catching all connections regardless of how they
# were originally stored.
query_search = """
MATCH (n)-[r]-(m)
WHERE n.name = $name
RETURN 
    n AS matchedNode,
    r AS relationship,
    m AS relatedNode
"""

In [ ]:
# Import the hybrid retrieval function (defined in retrieve/neo4j_pinecone.py).
# search_and_fetch() combines Pinecone vector search + Neo4j graph query:
#
#   1. Embeds `query_text` using OpenAI (same model used at ingest time —
#      this is critical: you must embed queries with the same model)
#   2. Queries Pinecone for the top-k most semantically similar chunks
#      (default top_k=2, configurable via parameter)
#   3. For each Pinecone result, reads metadata["name"] to get the entity name
#   4. Runs `query_search` (defined above) on Neo4j with that entity name
#   5. Returns a list of dicts, each containing:
#      {
#        "score":       cosine similarity (0–1, higher = more relevant),
#        "metadata":    the matching text chunk + section info,
#        "neo4j_nodes": all graph neighbors of the matched entity
#      }
#
# The combined result is far richer than plain vector search:
# the LLM can use both the specific chunk text AND the entity relationship map.
from retrieve.neo4j_pinecone import search_and_fetch

# Try changing the query_text to explore different results!
# Some ideas:
#   "What role did Talleyrand play in Napoleon's downfall?"
#   "Who were the commanders at the Battle of Waterloo?"
#   "What happened to Napoleon after Waterloo?"
search_and_fetch(query_search, query_text="Who killed Napoleon?")